In [44]:
import os
import pandas as pd
import numpy as np
import pickle
import yaml

from orbit.models import DLT

import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX
from pydlm import dlm, dynamic


In [45]:
df = pd.read_csv("../../data/scaler.csv")
df["Ngày"] = pd.to_datetime(df["Ngày"], format="%Y-%m-%d")
#df.set_index("Ngày", inplace=True)

In [46]:
df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,Cà phê Robusta nhân xô,0.904762,0.636364,0.153846,2025-05-09,128233.0
1,Cà phê Robusta nhân xô,0.952381,0.636364,0.153846,2025-05-09,128350.0
2,Cà phê Robusta nhân xô,0.238095,0.636364,0.153846,2025-05-09,128233.0
3,Cà phê Robusta nhân xô,0.476190,0.636364,0.153846,2025-05-09,128200.0
4,Cà phê Robusta nhân xô,0.571429,0.636364,0.153846,2025-05-09,128000.0


In [47]:
exog_cols = ["Thị_trường",	"Loại_giá",	"Nguồn"]

In [48]:
items = df["Tên_mặt_hàng"].unique()

In [49]:
for idx, item in enumerate(items):
    item_df = df[df["Tên_mặt_hàng"] == item]

    y = item_df["Giá"]
    exog_values = item_df[exog_cols]
    exog_values = np.array(exog_values)
    exog_component = dynamic(features=exog_values, discount=0.99, name='exog', w=1.0)

    model = dlm(y) + exog_component
    model.fit()

    with open(f"../../models/dlm_LBL/{idx}.pkl", "wb") as file:
        pickle.dump(model, file)

INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization fini

In [50]:
for idx, item in enumerate(items):
    item_df = df[df["Tên_mặt_hàng"] == item].sort_values("Ngày")

    y = item_df["Giá"]
    exog_values = item_df[exog_cols]

    model = SARIMAX(
        y,
        exog=exog_values,
        order=(2, 2, 2),     
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    # Save model
    with open(f"../../models/sarimax_LBL/{idx}.pkl", "wb") as file:
        pickle.dump(model, file)

d:\Apps\miniconda\envs\project3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Apps\miniconda\envs\project3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Apps\miniconda\envs\project3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Apps\miniconda\envs\project3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was

In [ ]:
model = DLT(
    response_col="Giá",
    date_col="Ngày",
    regressor_col=exog_cols,
    seed=42,
)

for idx, item in enumerate(items):
    item_df = df[df["Tên_mặt_hàng"] == item].sort_values("Ngày")
    
    # Group by date and take mean of prices (or sum, depending on your needs)
    item_df = item_df.groupby("Ngày").agg({
        "Giá": "mean",  # or "sum", "last", etc.
        **{col: "mean" for col in exog_cols}  # Handle exogenous variables
    }).reset_index()
    
    model.fit(item_df)
    
    with open(f"../../models/dlt_LBL/{idx}.pkl", "wb") as file:
        pickle.dump(model, file)